# 03 -- Position-Level Analysis (H4: entropy vs. accuracy)

Analysis-only notebook: no model training, no re-running the CNN. Reads the
canonical position table from `notebooks/data_pipeline.ipynb` (notebook 00)
and the already-saved 21bp predictions from `notebooks/01_main_experiment.ipynb`
(notebook 01), and asks whether positions with more inherently ambiguous
label distributions (higher `shannon_entropy`) are harder for the model to
predict correctly.

**Inputs verified against the actual files** (not assumed):
- `data/processed/position_table.csv` -- columns: `position_id, gene_number,
  cds_pos, cluster_id, n_instances, majority_subtype, n_distinct_subtypes,
  shannon_entropy`
- `data/processed/cluster_table.csv` -- columns: `cluster_id, n_positions,
  n_instances, majority_subtype, n_distinct_subtypes, shannon_entropy,
  hotspot_flag`
- `results/main/{hotspot,rare}/predictions.parquet` -- columns: `position_id,
  window_size, dataset, true_label, predicted_label, probabilities`

**Schema gaps found and resolved (confirmed with the user before building this
notebook):**
- `hotspot_flag` is not a column of `position_table.csv` -- it only exists in
  `cluster_table.csv`, keyed by `cluster_id`. Recovered by joining
  `position_table.cluster_id -> cluster_table.hotspot_flag`.
- `majority_subtype_freq` and `is_cpg` don't exist anywhere as precomputed
  columns. Both are computed here directly from
  `data/processed/tp53_mutation_dataset_w21.csv` (the combined 21bp instance
  table: `position_id, Sequence, MutationType`) and joined in. See the
  markdown cell right before they're computed for the window-size scoping
  note this implies.

Per the agreed framing: `is_cpg` is used here only as a visual grouping
variable in the entropy-vs-accuracy scatter plot. Any formal CpG-vs-non-CpG
significance test belongs in notebook 04, not here -- not duplicated.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import load_position_table, load_cluster_table, compute_is_cpg

POSITION_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'position_table.csv')
CLUSTER_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'cluster_table.csv')
COMBINED_W21_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv')

MAIN_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'main')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'position_analysis')
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = ['hotspot', 'rare']

print(f"Project root:      {PROJECT_ROOT}")
print(f"Position table:    {POSITION_TABLE_PATH}")
print(f"Cluster table:     {CLUSTER_TABLE_PATH}")
print(f"Combined 21bp data:{COMBINED_W21_PATH}")
print(f"Main results dir:  {MAIN_RESULTS_DIR}")
print(f"Output dir:        {RESULTS_DIR}")


Project root:      C:\Users\danya\Documents\projects\tp53_mutation_subtype
Position table:    C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\position_table.csv
Cluster table:     C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\cluster_table.csv
Combined 21bp data:C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w21.csv
Main results dir:  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main
Output dir:        C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis


## Load the canonical position table and join `hotspot_flag`

`hotspot_flag` lives on `cluster_table.csv`, not `position_table.csv` --
joined here via `cluster_id`.


In [ ]:
position_table = load_position_table(POSITION_TABLE_PATH)
cluster_table = load_cluster_table(CLUSTER_TABLE_PATH)

cluster_hotspot_flag = cluster_table.set_index('cluster_id')['hotspot_flag']
position_table = position_table.copy()
position_table['hotspot_flag'] = position_table['cluster_id'].map(cluster_hotspot_flag)

assert position_table['hotspot_flag'].isna().sum() == 0, (
    "Some position_ids have a cluster_id not found in cluster_table.csv"
)

print(f"Position table: {len(position_table):,} rows, "
      f"{position_table['hotspot_flag'].sum():,} hotspot / "
      f"{(~position_table['hotspot_flag']).sum():,} rare")
position_table.head()


## Compute `majority_subtype_freq` and `is_cpg` from the combined 21bp dataset

**Window-size scoping note:** `is_cpg` is defined from the 21bp window's
`Sequence` (centre base at index 10, next base at index 11 -- see
`compute_is_cpg` in `src/data.py`), so it is inherently a 21bp-specific
quantity. That's consistent with the rest of this notebook, which only uses
21bp predictions from notebook 01 -- but it means `is_cpg` (and, by
extension, anything derived from it) should not be reused as-is against
predictions from a different window size (e.g. the 11/51/101bp runs in
`02_window_ablation.ipynb`) without recomputing it from that window's own
`Sequence`.

`majority_subtype_freq` (proportion of a position's instances that are its
majority label) is window-size independent -- `Sequence` differs by window,
but `position_id`/`MutationType` don't -- so computing it from the 21bp
combined dataset is just a convenient, correct choice, not a scoping
constraint the way `is_cpg` is.


In [ ]:
combined_w21 = pd.read_csv(COMBINED_W21_PATH, dtype={'position_id': str})
required_cols = {'position_id', 'Sequence', 'MutationType'}
assert required_cols.issubset(combined_w21.columns), (
    f"{COMBINED_W21_PATH} is missing required columns: "
    f"{required_cols - set(combined_w21.columns)} (has: {combined_w21.columns.tolist()})"
)

# majority_subtype_freq: fraction of a position's instances equal to its majority label
majority_freq = (
    combined_w21.groupby('position_id')['MutationType']
    .agg(lambda s: s.value_counts().max() / len(s))
    .rename('majority_subtype_freq')
)

# QC: the majority label implied by this frequency computation must agree with
# position_table.majority_subtype (computed in notebook 00 from the 101bp basis) --
# MutationType is window-size independent, so these must match exactly. Uses the
# same tie-break rule as notebook 00 (pandas .mode()[0]) -- value_counts().idxmax()
# breaks count ties differently and produced 453 false-positive "mismatches" that
# were purely a tie-break artifact, not a real inconsistency (verified separately).
majority_label_check = (
    combined_w21.groupby('position_id')['MutationType']
    .agg(lambda s: s.mode()[0])
)
mismatch = position_table.set_index('position_id')['majority_subtype'] != majority_label_check.reindex(position_table['position_id']).values
n_mismatch = int(mismatch.sum())
assert n_mismatch == 0, (
    f"{n_mismatch} position(s) have a majority_subtype computed from the 21bp data "
    "that disagrees with position_table.majority_subtype -- MutationType should be "
    "window-size independent, this indicates a real inconsistency."
)
print(f"QC OK: majority label recomputed from {COMBINED_W21_PATH} agrees with "
      f"position_table.majority_subtype for all {len(position_table):,} positions.")

# is_cpg: one flag per position_id, from that position's (window-size-independent-per-locus) 21bp Sequence
seq_per_position = combined_w21.groupby('position_id')['Sequence'].first()
is_cpg_per_position = pd.Series(
    compute_is_cpg(seq_per_position.values, center_idx=10),
    index=seq_per_position.index,
    name='is_cpg',
)

position_table = position_table.merge(majority_freq, left_on='position_id', right_index=True, how='left')
position_table = position_table.merge(is_cpg_per_position, left_on='position_id', right_index=True, how='left')

assert position_table['majority_subtype_freq'].isna().sum() == 0, "Unmatched position_ids for majority_subtype_freq"
assert position_table['is_cpg'].isna().sum() == 0, "Unmatched position_ids for is_cpg"

print(f"\nAdded majority_subtype_freq (range {position_table['majority_subtype_freq'].min():.3f}"
      f"-{position_table['majority_subtype_freq'].max():.3f}) and is_cpg "
      f"({position_table['is_cpg'].sum():,} of {len(position_table):,} positions are CpG-context).")
position_table.head()


## Step 1: per-position accuracy from `predictions.parquet`

Group each dataset's predictions by `position_id`, compute accuracy
(`true_label == predicted_label`) per position, then join against the
enriched position table above.


In [5]:
per_dataset_analysis = {}

for name in DATASETS:
    pred_path = os.path.join(MAIN_RESULTS_DIR, name, 'predictions.parquet')
    preds = pd.read_parquet(pred_path)

    assert set(preds.columns) == {'position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities'}, (
        f"{pred_path}: unexpected columns {preds.columns.tolist()}"
    )
    assert (preds['window_size'] == 21).all(), f"{pred_path}: expected window_size == 21 everywhere"
    assert (preds['dataset'] == name).all(), f"{pred_path}: expected dataset == '{name}' everywhere"

    preds['correct'] = preds['true_label'] == preds['predicted_label']
    per_position_accuracy = preds.groupby('position_id')['correct'].mean().rename('accuracy')

    # Sanity check: every position_id in predictions must exist in the position table.
    pred_ids = set(per_position_accuracy.index)
    table_ids = set(position_table['position_id'])
    missing = pred_ids - table_ids
    if missing:
        raise AssertionError(
            f"{name}: {len(missing)} position_id(s) in predictions.parquet are missing from "
            f"position_table.csv -- join key mismatch between notebook 00 and notebook 01. "
            f"Examples: {sorted(missing)[:10]}"
        )
    print(f"{name}: {len(pred_ids):,} predicted positions, all present in position_table.csv. OK.")

    analysis_df = position_table.merge(per_position_accuracy, on='position_id', how='inner')

    # QC: hotspot_flag (from notebook 00's cluster table) should agree with which
    # predictions file ('dataset' column) this position came from.
    expected_flag = (name == 'hotspot')
    flag_mismatch = (analysis_df['hotspot_flag'] != expected_flag).sum()
    assert flag_mismatch == 0, (
        f"{name}: {flag_mismatch} position(s) have hotspot_flag != {expected_flag}, "
        "disagreeing with which predictions file they came from."
    )

    per_dataset_analysis[name] = analysis_df
    print(f"{name}: {len(analysis_df):,} positions in the joined analysis table\n")


hotspot: 1,136 predicted positions, all present in position_table.csv. OK.
hotspot: 1,136 positions in the joined analysis table

rare: 884 predicted positions, all present in position_table.csv. OK.
rare: 884 positions in the joined analysis table



## Persist the per-position table

Saved so downstream notebooks (`06_make_figures.ipynb`) can build final
figures purely by reading already-computed values, without re-deriving
per-position accuracy or `majority_subtype_freq` themselves.


In [ ]:
per_position_table = pd.concat([
    per_dataset_analysis[name][
        ['position_id', 'shannon_entropy', 'accuracy', 'majority_subtype_freq', 'is_cpg']
    ].assign(dataset=name)
    for name in DATASETS
], ignore_index=True)[['position_id', 'dataset', 'shannon_entropy', 'accuracy', 'majority_subtype_freq', 'is_cpg']]

per_position_table_path = os.path.join(RESULTS_DIR, 'per_position_table.csv')
per_position_table.to_csv(per_position_table_path, index=False)
print(f"Per-position table saved -> {per_position_table_path} ({len(per_position_table):,} rows)")
per_position_table.head()


## Step 2 (H4): Spearman correlation between `shannon_entropy` and per-position accuracy

Computed separately per dataset -- hotspot and rare are different test sets
and shouldn't be pooled.


In [7]:
spearman_results = {}

for name in DATASETS:
    df = per_dataset_analysis[name]
    corr, pval = spearmanr(df['shannon_entropy'], df['accuracy'])
    spearman_results[name] = {
        'correlation': float(corr),
        'p_value': float(pval),
        'n_positions': int(len(df)),
    }

spearman_path = os.path.join(RESULTS_DIR, 'spearman.json')
with open(spearman_path, 'w') as f:
    json.dump(spearman_results, f, indent=2)

print(f"Spearman results written -> {spearman_path}\n")
for name, res in spearman_results.items():
    print(f"{name}: Spearman rho={res['correlation']:.4f}  p={res['p_value']:.4g}  n={res['n_positions']:,}")


Spearman results written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\spearman.json

hotspot: Spearman rho=0.0318  p=0.2837  n=1,136
rare: Spearman rho=0.2752  p=7.9e-17  n=884


In [8]:
fig, ax = plt.subplots(figsize=(7, 5))
markers = {'hotspot': 'o', 'rare': 's'}
colors = {'hotspot': 'tab:blue', 'rare': 'tab:orange'}

for name in DATASETS:
    df = per_dataset_analysis[name]
    non_cpg = df[~df['is_cpg']]
    cpg = df[df['is_cpg']]

    ax.scatter(non_cpg['shannon_entropy'], non_cpg['accuracy'], marker=markers[name],
               color=colors[name], alpha=0.5, label=f'{name} (non-CpG)', s=25)
    ax.scatter(cpg['shannon_entropy'], cpg['accuracy'], marker=markers[name],
               facecolors='none', edgecolors=colors[name], linewidths=1.2,
               label=f'{name} (CpG)', s=45)

ax.set_xlabel('Shannon entropy of subtype distribution (position-level)')
ax.set_ylabel('Per-position accuracy')
ax.set_title('Entropy vs. accuracy (H4)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig_path = os.path.join(RESULTS_DIR, 'entropy_vs_accuracy.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\entropy_vs_accuracy.png


**Note:** `is_cpg` is shown above only as a visual grouping variable (open
vs. filled markers) to make the scatter more informative. No CpG-vs-non-CpG
significance test is computed in this notebook -- that comparison belongs in
notebook 04.


## Step 3: descriptive distributions (entropy, majority frequency)

In [9]:
fig, ax = plt.subplots(figsize=(7, 5))
bins = np.linspace(0, per_dataset_analysis['hotspot']['shannon_entropy'].max()
                    if per_dataset_analysis['hotspot']['shannon_entropy'].max()
                       >= per_dataset_analysis['rare']['shannon_entropy'].max()
                    else per_dataset_analysis['rare']['shannon_entropy'].max(), 25)

for name in DATASETS:
    ax.hist(per_dataset_analysis[name]['shannon_entropy'], bins=bins, alpha=0.5,
            density=True, label=name, color=colors[name])

ax.set_xlabel('Shannon entropy of subtype distribution')
ax.set_ylabel('Density')
ax.set_title('Distribution of position-level entropy')
ax.legend()
ax.grid(True, alpha=0.3)

fig_path = os.path.join(RESULTS_DIR, 'entropy_distribution.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\entropy_distribution.png


In [10]:
fig, ax = plt.subplots(figsize=(7, 5))
bins = np.linspace(0, 1, 25)

for name in DATASETS:
    ax.hist(per_dataset_analysis[name]['majority_subtype_freq'], bins=bins, alpha=0.5,
            density=True, label=name, color=colors[name])

ax.set_xlabel('Majority-subtype frequency (position-level)')
ax.set_ylabel('Density')
ax.set_title('Distribution of position-level majority-subtype frequency')
ax.legend()
ax.grid(True, alpha=0.3)

fig_path = os.path.join(RESULTS_DIR, 'majority_freq_distribution.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\majority_freq_distribution.png


## Step 4: `n_distinct_subtypes` summary

Connects to the effective-sample-size discussion already in the paper
(Section 3.3/4.1): how many positions are observed with only one subtype
ever, vs. genuine subtype diversity at that locus.


In [11]:
def bucket(n):
    if n <= 1:
        return '1'
    elif n == 2:
        return '2'
    else:
        return '3+'

n_distinct_rows = []
for name in DATASETS:
    df = per_dataset_analysis[name]
    buckets = df['n_distinct_subtypes'].apply(bucket).value_counts(normalize=True).reindex(['1', '2', '3+']).fillna(0)
    counts = df['n_distinct_subtypes'].apply(bucket).value_counts().reindex(['1', '2', '3+']).fillna(0).astype(int)
    n_distinct_rows.append({
        'dataset': name,
        'n_positions': len(df),
        'frac_1_subtype': buckets['1'],
        'frac_2_subtypes': buckets['2'],
        'frac_3plus_subtypes': buckets['3+'],
        'n_1_subtype': counts['1'],
        'n_2_subtypes': counts['2'],
        'n_3plus_subtypes': counts['3+'],
    })

n_distinct_df = pd.DataFrame(n_distinct_rows)
n_distinct_df


,dataset,n_positions,frac_1_subtype,frac_2_subtypes,frac_3plus_subtypes,n_1_subtype,n_2_subtypes,n_3plus_subtypes
0,hotspot,1136,0.082746,0.288732,0.628521,94,328,714
1,rare,884,0.533937,0.302036,0.164027,472,267,145


## Final printed summary (Spearman results + n_distinct_subtypes breakdown)

In [12]:
print("=" * 90)
print("SPEARMAN CORRELATION: shannon_entropy vs. per-position accuracy (H4)")
print("=" * 90)
for name, res in spearman_results.items():
    print(f"{name:>8}: rho={res['correlation']:+.4f}   p={res['p_value']:.4g}   n_positions={res['n_positions']:,}")

print("\n" + "=" * 90)
print("n_distinct_subtypes BREAKDOWN")
print("=" * 90)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print(n_distinct_df.to_string(index=False))

print("\nOutputs:")
print(f"  {os.path.join(RESULTS_DIR, 'spearman.json')}")
print(f"  {os.path.join(RESULTS_DIR, 'entropy_vs_accuracy.png')}")
print(f"  {os.path.join(RESULTS_DIR, 'entropy_distribution.png')}")
print(f"  {os.path.join(RESULTS_DIR, 'majority_freq_distribution.png')}")


SPEARMAN CORRELATION: shannon_entropy vs. per-position accuracy (H4)
 hotspot: rho=+0.0318   p=0.2837   n_positions=1,136
    rare: rho=+0.2752   p=7.9e-17   n_positions=884

n_distinct_subtypes BREAKDOWN
dataset  n_positions  frac_1_subtype  frac_2_subtypes  frac_3plus_subtypes  n_1_subtype  n_2_subtypes  n_3plus_subtypes
hotspot         1136        0.082746         0.288732             0.628521           94           328               714
   rare          884        0.533937         0.302036             0.164027          472           267               145

Outputs:
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\spearman.json
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\entropy_vs_accuracy.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\entropy_distribution.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\position_analysis\majority_freq_distribution